## Setup for Causal Inference Analysis


In [2]:
from pathlib import Path
import duckdb

data_dir = Path("../data").resolve()
# Connect to DuckDB database
con = duckdb.connect(str(data_dir / "project.duckdb"))

In [ ]:
# Begin by loading the tables we need from the DuckDB database into Pandas DataFrames. 
# We will load the `cleaned_data`, `games_clean`, `team_states` and `scope_states` tables.

cleaned_data = con.sql("SELECT * FROM cleaned_data").df()
print(f"cleaned_data shape: {cleaned_data.shape}")
games_clean = con.sql("SELECT * FROM games_clean").df()
print(f"games_clean shape: {games_clean.shape}")
team_states = con.sql("SELECT * FROM team_states").df()
print(f"team_states shape: {team_states.shape}")
scope_states = con.sql("SELECT * FROM scope_states").df()
print(f"scope_states shape: {scope_states.shape}")

In [ ]:
# Create combined DataFrame with cleaned_data and scope_states
combined_df = con.sql("""
    WITH state_years AS (
        SELECT
            state,
            year
        FROM scope_states
        CROSS JOIN range(2016, 2026) AS y(year)
    ),
    outcomes AS (
        SELECT
            source_state AS state,
            source_year AS year,
            COUNT(DISTINCT incident_id) AS dv_incidents
        FROM cleaned_data
        GROUP BY source_state, source_year
    )
    SELECT
        sy.state,
        sy.year,
        COALESCE(SUM(o.dv_incidents), 0) AS dv_incidents,
        s.scope_group,
        s.legal_date
    FROM state_years sy
    LEFT JOIN outcomes o
        ON sy.state = o.state
       AND sy.year = o.year
    JOIN scope_states s USING (state)
    ORDER BY state, year
""").df()

# Define treatment variable based on next calendar year after legal_date
combined_df['trt'] = (combined_df['legal_date'].notnull() & (combined_df['year'] > combined_df['legal_date'].dt.year))